# Key findings, reproduced from the API

The findings `docs/findings.md` quotes, read from the running backend through the documented
`/insights` and `/trends` endpoints only (`docs/agents.md`, "The endpoints added for agents" and "The shapes of the generated endpoints"): no database access, no hidden
computation. Every figure below is a value an endpoint returned, or a sum, difference or ratio of such values.

**Run it:** start the backend (`python run.py --backend`, port 8000), then run all cells. Another backend:
set `LUX_API` (e.g. `LUX_API=http://localhost:8001`). Needs pandas (in `requirements.txt`) and Jupyter (`pip install jupyter`, not in the requirements); the saved outputs are from 15 October 2026.

Data: Eurostat ds-045409, reporter Luxembourg, nominal euros. 2026 months are provisional.

1. The 2025 export rise and the smartphone line
2. The thread: where the smartphones came from and went to
3. Four lasting shifts since 1999 (change points)
4. The 2026 forecast and its two tests
5. Where Luxembourg is specialised (revealed comparative advantage)
6. Luxembourg among its EU peers

In [1]:
import json
import os
import urllib.parse
import urllib.request

import pandas as pd

API = os.environ.get("LUX_API", "http://localhost:8000")
CALLS = []


def get(endpoint, **params):
    """GET a documented endpoint; remembers the call so the notebook lists its sources."""
    url = endpoint + ("?" + urllib.parse.urlencode(params) if params else "")
    CALLS.append(url)
    with urllib.request.urlopen(API + url, timeout=180) as r:
        return json.loads(r.read().decode("utf-8"))


def eur(v):
    return f"€{v / 1e9:.2f} bn" if abs(v) >= 1e9 else f"€{v / 1e6:.1f} m"


def pct(v, signed=False):
    return f"{v * 100:+.1f}%" if signed else f"{v * 100:.1f}%"


meta = get("/insights/meta")
print(meta["title"])
print("years", meta["periods"]["A"][0], "to", meta["periods"]["A"][-1], "| months",
      meta["periods"]["M"][0], "to", meta["periods"]["M"][-1], "| complete until", meta["complete_until"])

Luxembourg external trade in goods (Eurostat ds-045409)
years 1999 to 2025 | months 1999-01 to 2026-06 | complete until 2025-12


## 1. The 2025 export rise and the smartphone line

Exports rose from 2024 to 2025; the largest mover at CN8 level (8 digits) is one line, smartphones
(CN8 85171300). Without that line, the rise is much smaller.

In [2]:
years = get("/insights/series", flow="EXPORT", freq="A", start="2024", end="2025")
before, after = years[0]["value"], years[-1]["value"]
rise = after - before

movers = get("/insights/movers", by="product", period="2025", previous="2024", flow="EXPORT", level=4)
top = movers["up"][0]
phones = get("/insights/series", flow="EXPORT", freq="A", start="2024", end="2025", product=top["code"])
p_before, p_after = phones[0]["value"], phones[-1]["value"]

print(f"Exports 2024 -> 2025: {eur(before)} -> {eur(after)}, {eur(rise)} ({pct(after / before - 1, True)})")
print(f"Largest CN8 mover: {top['code']} {top['name']}: {eur(top['previous'])} -> {eur(top['value'])}")
print(f"Its share of the rise: {pct(top['change'] / rise)}")
print(f"Exports without it: {pct((after - p_after) / (before - p_before) - 1, True)}")
pd.DataFrame(movers["up"][:5])[["code", "name", "previous", "value", "change"]]

Exports 2024 -> 2025: €14.97 bn -> €16.25 bn, €1.29 bn (+8.6%)
Largest CN8 mover: 85171300 Smartphones for wireless networks: €42.3 m -> €898.3 m
Its share of the rise: 66.5%
Exports without it: +2.9%


,code,name,previous,value,change
0,85171300,Smartphones for wireless networks,42293544.0,898331987.0,856038443.0
1,71081200,"Gold, incl. gold plated with platinum, unwroug...",45627308.0,132486756.0,86859448.0
2,84111210,Turbojets of a thrust > 25 kN but <= 44 kN,0.0,61173458.0,61173458.0
3,84111230,Turbojets of a thrust > 44 kN but <= 132 kN,32695554.0,88849603.0,56154049.0
4,87038090,Motor cars and other motor vehicles principall...,48036404.0,96080534.0,48044130.0


## 2. The thread: where the smartphones came from and went to

The same CN8 line on both sides of Luxembourg: who supplied it and who bought it. Shares of one line have
that line as their base; the last share is the line against everything bought from China.

In [3]:
line = top["code"]
sold = get("/insights/ranking", by="partner", period="2025", flow="EXPORT", product=line)
bought = get("/insights/ranking", by="partner", period="2025", flow="IMPORT", product=line)
fr = next(i for i in sold["items"] if i["code"] == "FR")
cn = next(i for i in bought["items"] if i["code"] == "CN")
china = get("/insights/series", flow="IMPORT", freq="A", partner="CN", start="2025", end="2025")[-1]["value"]

print(f"Sold to France: {eur(fr['value'])}, {pct(fr['share'])} of the line's exports")
print(f"Bought from China: {eur(cn['value'])}, {pct(cn['share'])} of the line's imports")
print(f"Everything bought from China in 2025: {eur(china)}; the line is {pct(cn['value'] / china)} of it")

Sold to France: €770.2 m, 85.7% of the line's exports
Bought from China: €613.4 m, 87.5% of the line's imports
Everything bought from China in 2025: €761.3 m; the line is 80.6% of it


## 3. Four lasting shifts since 1999 (change points)

PELT change points on the monthly export series since January 1999 (the trend finder's settings: penalty 1.5,
at least 12 months per regime). The levels are typical months before and after each break.

In [4]:
last_month = meta["periods"]["M"][-1]
analysis = get("/trends/analyze", flow="EXPORT", freq="M", start="1999-01", end=last_month, penalty=1.5, min_size=12)
breaks = pd.DataFrame([{"month": c["period"], "before (a month)": eur(c["before_mean"]),
                        "after (a month)": eur(c["after_mean"]), "change": pct(c["relative_change"], True)}
                       for c in analysis["changepoints"]])
print(analysis["seasonality_summary"])
breaks

A moderate repeating yearly pattern: Oct is typically the highest point (much higher than usual), Aug the lowest (much lower than usual).


,month,before (a month),after (a month),change
0,2001-02,€766.4 m,€912.2 m,+19.0%
1,2005-09,€1.17 bn,€1.55 bn,+32.7%
2,2009-01,€1.46 bn,€1.27 bn,-12.4%
3,2021-12,€1.17 bn,€1.37 bn,+17.5%


## 4. The 2026 forecast and its two tests

The default method (the backtest's winner) on the monthly series, summed to the year; an 80% conformal
interval; scenarios for the smartphone line. Two different tests, both true:

- **the rolling backtest**: at each of the last quarter ends, fitted on the months before it, scored on the
  12 months after, against the seasonal naive forecast (the same month a year earlier);
- **the single last full year**: 2025, forecast from December 2024, against repeating 2024.

In [5]:
fc = get("/insights/forecast", flow="EXPORT", freq="M", horizon=12, interval=0.8, aggregate="A")
row = fc["rows"][-1]
bt = fc["backtest"]
print(f"Exports {row['period']}: {eur(row['value'])}, 80% interval {eur(row['low'])} to {eur(row['high'])}; "
      f"{row['observed_months']} months already in at {eur(row['observed'])}")
for s in fc.get("scenarios", []):
    v = next(r["value"] for r in s["rows"] if r["period"] == row["period"])
    print(f"  if the smartphone line {s['case']} (since {s['since']}): {eur(v)}")
print(f"Method: {fc['method_label']}")
print(f"Test 1: at the {bt['origins']} quarter ends from {bt['first_origin']} to {bt['last_origin']}, "
      f"{pct(bt['skill_h1'])} smaller errors than the naive forecast 1 month ahead, {pct(bt['skill_h12'])} 12 months "
      f"ahead; the band held {pct(bt['coverage'])} of the months")
print(f"Test 2: {bt['period']} alone, off by {pct(bt['mape'])} against {pct(bt['naive_mape'])} for repeating the year "
      f"before ({'lost' if bt['mape'] > bt['naive_mape'] else 'won'})")

Exports 2026: €17.34 bn, 80% interval €16.65 bn to €18.10 bn; 6 months already in at €8.60 bn
  if the smartphone line stops (since 2025-09): €16.73 bn
  if the smartphone line continues (since 2025-09): €17.04 bn
Method: the average of Holt-Winters exponential smoothing and gradient boosting trained on every HS2 chapter
Test 1: at the 20 quarter ends from 2021-06 to 2026-03, 49.2% smaller errors than the naive forecast 1 month ahead, 3.0% 12 months ahead; the band held 81.1% of the months
Test 2: 2025 alone, off by 9.9% against 7.9% for repeating the year before (lost)


In [6]:
# Every method's rolling backtest, the neural network generated by BESSER included (method=nn).
nn = get("/insights/forecast", flow="EXPORT", freq="M", horizon=12, interval=0.8, aggregate="A", method="nn")
methods = {m["method"]: m for m in bt["methods"]}
for m in nn["backtest"]["methods"]:
    methods.setdefault(m["method"], m)
naive = methods["snaive"]
table = pd.DataFrame([{"method": k, "default": bool(m.get("chosen")) and k != "nn",
                       **{f"beats naive, {h} m": pct(1 - m[f"mase_h{h}"] / naive[f"mase_h{h}"], True) for h in (1, 3, 6, 12)},
                       "MASE": round(m["mase"], 3), "band held": pct(m["coverage"])}
                      for k, m in sorted(methods.items(), key=lambda kv: kv[1]["mase"])])
table

,method,default,"beats naive, 1 m","beats naive, 3 m","beats naive, 6 m","beats naive, 12 m",MASE,band held
0,gbm,False,+48.1%,+51.3%,+40.5%,+9.5%,0.629,87.1%
1,ensemble,True,+49.2%,+53.2%,+36.4%,+3.0%,0.644,81.1%
2,nn,False,+40.5%,+33.9%,+25.0%,-17.3%,0.733,84.1%
3,ets,False,+33.0%,+46.9%,+30.5%,-5.9%,0.733,85.6%
4,theta,False,+31.3%,+46.5%,+30.5%,-5.4%,0.740,86.4%
5,snaive,False,+0.0%,+0.0%,+0.0%,+0.0%,0.916,90.9%


## 5. Where Luxembourg is specialised (revealed comparative advantage)

RCA = a chapter's share in Luxembourg's exports divided by its share in the exports of the 26 other EU
countries. Above 1: Luxembourg is specialised in it.

In [7]:
spec = get("/insights/specialisation", by="product", period="2025", flow="EXPORT")
print(f"{len(spec['peers'])} peers")
pd.DataFrame([{"chapter": i["code"], "name": i["name"][:60], "RCA": round(i["rca"], 1), "LU exports": eur(i["value"])}
              for i in spec["items"][:8]])

26 peers


,chapter,name,RCA,LU exports
0,56,"Wadding, felt and nonwovens; special yarns; tw...",17.5,€434.2 m
1,59,"Impregnated, coated, covered or laminated text...",10.3,€201.6 m
2,40,Rubber and articles thereof,5.0,€851.8 m
3,81,Other base metals; cermets; articles thereof,4.1,€62.4 m
4,04,Dairy produce; birds' eggs; natural honey; edi...,4.0,€720.7 m
5,72,Iron and steel,3.6,€1.12 bn
6,70,Glass and glassware,2.9,€210.0 m
7,24,Tobacco and manufactured tobacco substitutes,2.7,€173.3 m


## 6. Luxembourg among its EU peers

Growth of exports of goods from 2015 to 2025, nominal euros, every EU reporter.

In [8]:
peers = get("/insights/compare/records", by="reporter", period="2025", base="2015", flow="EXPORT")
lu = next(i for i, r in enumerate(peers) if r["is_focus"])
print(f"Luxembourg: {peers[lu]['growth']:+.1f}%, {lu + 1} of {len(peers)}")
for code_ in ("BE", "DE"):
    r = next(r for r in peers if r["code"] == code_)
    print(f"{r['name']}: {r['growth']:+.1f}%")
pd.DataFrame(peers)[["code", "name", "growth"]].tail(5)

Luxembourg: +5.1%, 27 of 27
Belgium: +41.4%
Germany: +30.7%


,code,name,growth
22,BE,Belgium,41.4
23,FI,Finland,37.7
24,FR,France,32.8
25,DE,Germany,30.7
26,LU,Luxembourg ◀,5.1


## Sources

Every call this notebook made, in order (relative to the backend's URL).

In [9]:
for c in dict.fromkeys(CALLS):
    print(c)

/insights/meta
/insights/series?flow=EXPORT&freq=A&start=2024&end=2025
/insights/movers?by=product&period=2025&previous=2024&flow=EXPORT&level=4
/insights/series?flow=EXPORT&freq=A&start=2024&end=2025&product=85171300
/insights/ranking?by=partner&period=2025&flow=EXPORT&product=85171300
/insights/ranking?by=partner&period=2025&flow=IMPORT&product=85171300
/insights/series?flow=IMPORT&freq=A&partner=CN&start=2025&end=2025
/trends/analyze?flow=EXPORT&freq=M&start=1999-01&end=2026-06&penalty=1.5&min_size=12
/insights/forecast?flow=EXPORT&freq=M&horizon=12&interval=0.8&aggregate=A
/insights/forecast?flow=EXPORT&freq=M&horizon=12&interval=0.8&aggregate=A&method=nn
/insights/specialisation?by=product&period=2025&flow=EXPORT
/insights/compare/records?by=reporter&period=2025&base=2015&flow=EXPORT
